# **Cell 1  Imports & Drive Mount**

In [ ]:
from google.colab import drive
import os, glob, json
import numpy as np, pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


# ***Cell 2 Locate & Unzip Dataset***

In [ ]:
zip_files = glob.glob('/content/drive/MyDrive/**/cleaned_dataset*.zip', recursive=True)
if zip_files:
    os.system(f'unzip -o -q "{zip_files[0]}" -d /content/')

meta_path = next(
    os.path.join(r, f) for r, _, fs in os.walk('/content') for f in fs
    if f == 'metadata.csv' and 'drive' not in r
)
data_dir = os.path.join(os.path.dirname(meta_path), 'data')

print('metadata:', meta_path)
print('data_dir:', data_dir)

metadata: /content/cleaned_dataset/metadata.csv
data_dir: /content/cleaned_dataset/data


# ***Cell 3 Load Metadata & Compute SOH / RUL***

In [ ]:
df = pd.read_csv(meta_path)
df['Capacity'] = pd.to_numeric(
    df['Capacity'].astype(str).str.replace(r'[\[\]]', '', regex=True), errors='coerce'
)
dis_df = (df[(df['type'] == 'discharge') & (df['Capacity'] > 0.05)]
          .sort_values(['battery_id', 'test_id']).copy())

dis_df['SOH'] = dis_df['Capacity'] / dis_df.groupby('battery_id')['Capacity'].transform('first')

def add_rul(g):
    eol_mask = g['Capacity'] <= 0.7 * g['Capacity'].iloc[0]
    eol_test_id = g.loc[eol_mask, 'test_id'].iloc[0] if eol_mask.any() else g['test_id'].max()
    g['RUL'] = (eol_test_id - g['test_id']).clip(lower=0)
    return g

dis_df = dis_df.groupby('battery_id', group_keys=False).apply(add_rul)

print(dis_df.shape)
dis_df.head()

(2714, 12)


/tmp/ipykernel_6409/2636989889.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dis_df = dis_df.groupby('battery_id', group_keys=False).apply(add_rul)


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,SOH,RUL
5121,discharge,[2.0080e+03 4.0000e+00 2.0000e+00 1.5000e+01 2...,24,B0005,1,5122,05122.csv,1.856487,NaN,NaN,1.000000,590
5123,discharge,[2.0080e+03 4.0000e+00 2.0000e+00 1.9000e+01 4...,24,B0005,3,5124,05124.csv,1.846327,NaN,NaN,0.994527,588
5125,discharge,[2.008e+03 4.000e+00 3.000e+00 0.000e+00 1.000...,24,B0005,5,5126,05126.csv,1.835349,NaN,NaN,0.988614,586
5127,discharge,[2008. 4. 3. 4. 16. ...,24,B0005,7,5128,05128.csv,1.835263,NaN,NaN,0.988567,584
5129,discharge,[2008. 4. 3. 8. 33. ...,24,B0005,9,5130,05130.csv,1.834646,NaN,NaN,0.988235,582


# ***Cell 4 Feature Extraction Function***

In [ ]:
def extract_features(row):
    f = os.path.join(data_dir, row['filename'])
    if not os.path.exists(f):
        return None
    try:
        raw = pd.read_csv(f)
        if len(raw) < 10 or 'Time' not in raw or raw['Time'].iloc[-1] < 60:
            return None
        w = raw[raw['Time'] <= 500]
        if len(w) < 5:
            return None
        i3 = min(3, len(w) - 1)
        dt = w['Time'].iloc[-1] - w['Time'].iloc[0] + 1e-6
        return {
            'battery_id': row['battery_id'], 'test_id': int(row['test_id']),
            'ambient_temperature': row['ambient_temperature'],
            'feat_R_est': (w['Voltage_measured'].iloc[0] - w['Voltage_measured'].iloc[i3])
                          / (abs(w['Current_measured'].iloc[i3]) + 1e-6),
            'feat_dV_dt': (w['Voltage_measured'].iloc[-1] - w['Voltage_measured'].iloc[0]) / dt,
            'feat_dT_dt': (w['Temperature_measured'].iloc[-1] - w['Temperature_measured'].iloc[0]) / dt,
            'feat_V_mean': float(w['Voltage_measured'].mean()),
            'feat_T_mean': float(w['Temperature_measured'].mean()),
            'target_capacity': row['Capacity'], 'target_SOH': row['SOH'], 'target_RUL': row['RUL']
        }
    except Exception:
        return None

# ***Cell 5 Run Extraction (Build data DataFrame)***

In [ ]:
records = [r for r in (extract_features(row) for _, row in dis_df.iterrows()) if r]
data = pd.DataFrame(records)

print(data.shape)
data.head()

(2705, 11)


,battery_id,test_id,ambient_temperature,feat_R_est,feat_dV_dt,feat_dT_dt,feat_V_mean,feat_T_mean,target_capacity,target_SOH,target_RUL
0,B0005,1,24,0.119055,-0.000845,0.008845,3.875140,26.463501,1.856487,1.000000,590
1,B0005,3,24,0.115632,-0.000830,0.008620,3.880955,26.768016,1.846327,0.994527,588
2,B0005,5,24,0.114162,-0.000823,0.008498,3.882830,26.789512,1.835349,0.988614,586
3,B0005,7,24,0.113519,-0.000820,0.008445,3.884857,26.683526,1.835263,0.988567,584
4,B0005,9,24,0.113352,-0.000818,0.008431,3.885669,26.549100,1.834646,0.988235,582


# ***Cell 6 Split Functions***

In [ ]:
def cell_split(df, val_cells, test_cells):
    is_val, is_test = df['battery_id'].isin(val_cells), df['battery_id'].isin(test_cells)
    return df[~is_val & ~is_test], df[is_val], df[is_test]

def temporal_split(df, ratios=(0.6, 0.8)):
    parts = {'train': [], 'val': [], 'test': []}
    for _, g in df.groupby('battery_id'):
        n = len(g)
        parts['train'].append(g.iloc[:int(n*ratios[0])])
        parts['val'].append(g.iloc[int(n*ratios[0]):int(n*ratios[1])])
        parts['test'].append(g.iloc[int(n*ratios[1]):])
    return (pd.concat(parts['train']), pd.concat(parts['val']), pd.concat(parts['test']))

# ***Cell 7 Apply Splits***

In [ ]:
test_cells = ['B0018', 'B0028', 'B0032', 'B0036', 'B0040', 'B0043', 'B0044', 'B0051', 'B0056']
val_cells  = ['B0007', 'B0027', 'B0031', 'B0034', 'B0039', 'B0042', 'B0048', 'B0050', 'B0054']

cell_train, cell_val, cell_test = cell_split(data, val_cells, test_cells)
temp_train, temp_val, temp_test = temporal_split(data)

print('cell:', len(cell_train), len(cell_val), len(cell_test))
print('temporal:', len(temp_train), len(temp_val), len(temp_test))

cell: 1141 778 786
temporal: 1610 542 553


# ***Cell 8 Save CSVs + Summary JSON***

In [ ]:
outputs = {
    'cell_train': cell_train, 'cell_val': cell_val, 'cell_test': cell_test,
    'temporal_train': temp_train, 'temporal_val': temp_val, 'temporal_test': temp_test
}
for name, d in outputs.items():
    d.to_csv(f'/content/{name}.csv', index=False)

summary = {
    'total_samples': len(data),
    'cells_count': int(data['battery_id'].nunique()),
    'cell_split': {k: len(outputs[f'cell_{k}']) for k in ['train', 'val', 'test']},
    'temporal_split': {k: len(outputs[f'temporal_{k}']) for k in ['train', 'val', 'test']}
}
with open('/content/data_prep_frozen_config.json', 'w') as fp:
    json.dump(summary, fp, indent=4)

print('SUCCESS:', json.dumps(summary, indent=2))

SUCCESS: {
  "total_samples": 2705,
  "cells_count": 34,
  "cell_split": {
    "train": 1141,
    "val": 778,
    "test": 786
  },
  "temporal_split": {
    "train": 1610,
    "val": 542,
    "test": 553
  }
}
